In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Video, clear_output

import cv2
import mediapipe as mp
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

#### Чтение данных

In [21]:
data = pd.read_csv('data/annotations.csv', sep='\t', on_bad_lines='skip')
data

,attachment_id,text,user_id,height,width,length,train,begin,end
0,44e8d2a0-7e01-450b-90b0-beb7400d2c1e,Ё,185bd3a81d9d618518d10abebf0d17a8,1920,1080,156.0,True,36,112
1,df5b08f0-41d1-4572-889c-8b893e71069b,А,185bd3a81d9d618518d10abebf0d17a8,1920,1080,150.0,True,36,76
2,17f53df4-c467-4aff-9f48-20687b63d49a,Р,185bd3a81d9d618518d10abebf0d17a8,1920,1080,133.0,True,40,97
3,e3add916-c708-4339-ad98-7e2740be29e9,Е,185bd3a81d9d618518d10abebf0d17a8,1920,1080,144.0,True,43,107
4,bd7272ed-1850-48f1-a2a8-c8fed523dc37,Ч,185bd3a81d9d618518d10abebf0d17a8,1920,1080,96.0,True,20,70
...,...,...,...,...,...,...,...,...,...
20395,nodca88242-2bc7-4a77-9d14-103aa1dacbd6,no_event,0041ec866777f12c384b64d8cd636277,1920,1080,42.0,False,0,42
20396,no7a7812b1-ae64-4402-9ebc-5b947edbd021,no_event,f5b82a9c82f6d870ec253e4c3fa96d83,1280,720,32.0,False,0,32
20397,no62a4df76-b48d-4f61-b6e8-bc4a1eb0cb61,no_event,4299b8ccf39ace57287b463fbe4a489b,1920,960,32.0,False,0,32
20398,no388a3f7c-3594-4332-bc78-b5b53190301d,no_event,c80b4e57f158f28299b2a89694c42329,1920,1080,41.0,False,0,41


In [ ]:
new_data = data
new_data

,attachment_id,text,user_id,height,width,length,train,begin,end
0,44e8d2a0-7e01-450b-90b0-beb7400d2c1e,Ё,185bd3a81d9d618518d10abebf0d17a8,1920,1080,156.0,True,36,112
1,df5b08f0-41d1-4572-889c-8b893e71069b,А,185bd3a81d9d618518d10abebf0d17a8,1920,1080,150.0,True,36,76
2,17f53df4-c467-4aff-9f48-20687b63d49a,Р,185bd3a81d9d618518d10abebf0d17a8,1920,1080,133.0,True,40,97
3,e3add916-c708-4339-ad98-7e2740be29e9,Е,185bd3a81d9d618518d10abebf0d17a8,1920,1080,144.0,True,43,107
4,bd7272ed-1850-48f1-a2a8-c8fed523dc37,Ч,185bd3a81d9d618518d10abebf0d17a8,1920,1080,96.0,True,20,70
...,...,...,...,...,...,...,...,...,...
20395,nodca88242-2bc7-4a77-9d14-103aa1dacbd6,no_event,0041ec866777f12c384b64d8cd636277,1920,1080,42.0,False,0,42
20396,no7a7812b1-ae64-4402-9ebc-5b947edbd021,no_event,f5b82a9c82f6d870ec253e4c3fa96d83,1280,720,32.0,False,0,32
20397,no62a4df76-b48d-4f61-b6e8-bc4a1eb0cb61,no_event,4299b8ccf39ace57287b463fbe4a489b,1920,960,32.0,False,0,32
20398,no388a3f7c-3594-4332-bc78-b5b53190301d,no_event,c80b4e57f158f28299b2a89694c42329,1920,1080,41.0,False,0,41


In [ ]:
all_class_counts = new_data['text'].value_counts()
top_classes = all_class_counts.index.tolist()
new_data = new_data[new_data['text'].isin(top_classes)]

class_names = sorted(top_classes)
label2idx = {label: idx for idx, label in enumerate(class_names)}
len(class_names)

1001

#### Инициализация параметров для MediaPipe

In [24]:
# Параметры
source_directory = 'data' # базовая папка
train_dir = os.path.join(source_directory, 'train')
test_dir = os.path.join(source_directory, 'test')
video_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.flv', '.wmv'}
SEQUENCE_LENGTH = 48
MAX_HANDS = 2
NUM_LANDMARKS = 21 #количество точек для каждой руки
NUM_FEATURES_PER_LANDMARK_HAND = 3 #количество координат для каждой точки
NUM_FEATURES = MAX_HANDS * NUM_LANDMARKS * NUM_FEATURES_PER_LANDMARK_HAND  #21 точка по 3 координаты

NUM_FEATURES_PER_LANDMARK = 2
LIPS_IDX = [61, 37, 0, 267, 291, 405, 17, 181] #точки губ
NUM_FEATURES_FACE = len(LIPS_IDX)

POSE_IDX = [16, 14, 12, 11, 13, 15, 0] #точки рук + нос
NUM_FEATURES_POSE = len(POSE_IDX)

NUM_FEATURES_ALL = NUM_FEATURES + NUM_FEATURES_FACE * NUM_FEATURES_PER_LANDMARK + NUM_FEATURES_POSE * NUM_FEATURES_PER_LANDMARK


In [31]:
!curl -L -o hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!curl -L -o face_landmarker_v2_with_blendshapes.task -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!curl -L -o pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
 48 7635k   48 3711k    0     0  3632k      0  0:00:02  0:00:01  0:00:01 3638k
100 7635k  100 7635k    0     0  5087k      0  0:00:01  0:00:01 --:--:-- 5093k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
 24 3670k   24  893k    0     0  1867k      0  0:00:01 --:--:--  0:00:01 1872k
100 3670k  100 3670k    0     0  4023k      0 --:--:-- --:--:-- --:--:-- 4033k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   

In [25]:
base_options_hands = python.BaseOptions(model_asset_path='hand_landmarker.task')
options_hands = vision.HandLandmarkerOptions(
    base_options=base_options_hands,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.3,
)

In [26]:
base_options_face = python.BaseOptions(model_asset_path='face_landmarker_v2_with_blendshapes.task')
options_face = vision.FaceLandmarkerOptions(base_options=base_options_face,
                                       running_mode=vision.RunningMode.VIDEO,
                                       num_faces=1,
                                       min_face_detection_confidence=0.3)

In [27]:
base_options_pose = python.BaseOptions(model_asset_path='pose_landmarker.task')
options_pose = vision.PoseLandmarkerOptions(
    base_options=base_options_pose,
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.3)

#### Извлечение координат ключевых точек фреймов из видео и сохранение их в файл

In [ ]:
def find_video_path(attachment_id, is_train=True):
    folder = train_dir if is_train else test_dir
    for ext in video_extensions:
        path = os.path.join(folder, f"{attachment_id}{ext}")
        if os.path.exists(path):
            return path
    return None

def extract_sequence_from_video(video_path):
    detector_hands = vision.HandLandmarker.create_from_options(options_hands)
    detector_face = vision.FaceLandmarker.create_from_options(options_face)
    detector_pose = vision.PoseLandmarker.create_from_options(options_pose)

    data_is_arr =[]

    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_count = 0
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30  # защита от 0

    step = max(1, int(fps / 15))  # Автоподбор шага для ~15 кадров/сек

    while cap.isOpened():
        data_is_present = [0, 0, 0, 0] #hands, face, pose
        frame_count += 1
        ret, frame = cap.read()
        if not ret:
            break

        # Пропускаем кадры для оптимизации
        if frame_count % step != 0:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=frame_rgb
        )

        timestamp_ms = int(frame_count * 1000 / fps) + 1

        results_hand = detector_hands.detect_for_video(mp_image, timestamp_ms)
        results_face = detector_face.detect_for_video(mp_image, timestamp_ms)
        results_pose = detector_pose.detect_for_video(mp_image, timestamp_ms)
        landmarks = []

        if results_hand.hand_landmarks:
            landmarks_hands = []
            landmarks_hand = []
            hand_idx = 0
            for hand in results_hand.hand_landmarks[:MAX_HANDS]:
              data_is_present[hand_idx] = 1
              hand_idx += 1
              for lm in hand:
                landmarks_hands.extend([lm.x, lm.y, lm.z])

            while len(landmarks_hands) < NUM_FEATURES:
              landmarks_hands.extend([np.nan] * NUM_LANDMARKS * NUM_FEATURES_PER_LANDMARK_HAND)
            landmarks.extend(landmarks_hands)
        else:
            landmarks.extend([np.nan] * NUM_FEATURES)

        if results_face.face_landmarks:
            landmarks_lips = []
            data_is_present[2] = 1
            for idx in LIPS_IDX:
                lm = results_face.face_landmarks[0][idx]
                landmarks.extend([lm.x, lm.y])
        else:
            landmarks.extend([np.nan] * NUM_FEATURES_FACE * NUM_FEATURES_PER_LANDMARK)


        if results_pose.pose_landmarks:
            landmarks_pose = []
            data_is_present[-1] = 1
            for idx in POSE_IDX:
                lm = results_pose.pose_landmarks[0][idx]
                landmarks.extend([lm.x, lm.y])
        else:
            landmarks.extend([np.nan] * NUM_FEATURES_POSE * NUM_FEATURES_PER_LANDMARK)

        if len(landmarks) != NUM_FEATURES_ALL:
          print("WRONG FRAME")
          print("len(landmarks):", len(landmarks))
          print("expected:", NUM_FEATURES_ALL)
          raise ValueError("Feature length mismatch")

        frames.append(landmarks)
        data_is_arr.append(data_is_present)

    cap.release()

    detector_hands.close()
    detector_face.close()
    detector_pose.close()

    first_action_idx = 0
    data_is_there = False

    for i, f in enumerate(frames):
        if np.any(np.array(data_is_arr[:2]) != 0):
            first_action_idx = i
            data_is_there = True
            break

    end_idx = first_action_idx + SEQUENCE_LENGTH

    sequence = np.array(
    frames[first_action_idx:end_idx],
    dtype=np.float32
)

    if len(sequence) < SEQUENCE_LENGTH:
        padding = np.full((SEQUENCE_LENGTH - len(sequence), NUM_FEATURES_ALL), np.nan)
        sequence = np.vstack([sequence, padding])

    sequence[np.isnan(sequence)] = 0
    return (data_is_there, sequence)



In [ ]:
# Основной цикл обработки видео
X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

os.makedirs("data/dataset_train_proc", exist_ok=True)
os.makedirs("data/dataset_test_proc", exist_ok=True)
txt_path = "data/problem_files.txt"
os.makedirs(os.path.dirname(txt_path), exist_ok=True)

for idx, row in tqdm(new_data.iterrows(), total=len(new_data)):
  attachment_id = row['attachment_id']
  label = row['text']

  video_path = find_video_path(attachment_id, is_train=True)
  if video_path:
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    if os.path.exists(f"data/dataset_train_proc/{video_name}.npy"):
      continue
    check, seq = extract_sequence_from_video(video_path)
    if not check:
      print('Не удалось преобразовать видео - ' + video_path)
      with open(txt_path, "a", encoding="utf-8") as f:
        f.write(video_name + "\n")
    X_train_list.append(seq)
    y_train_list.append(label2idx[label])
    np.save(f"data/dataset_train_proc/{video_name}.npy", seq)
    continue

  video_path = find_video_path(attachment_id, is_train=False)
  if video_path:
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    if os.path.exists(f"data/dataset_test_proc/{video_name}.npy"):
      continue
    check, seq = extract_sequence_from_video(video_path)
    if not check:
      print('Не удалось преобразовать видео - ' + video_path)
      with open(txt_path, "a", encoding="utf-8") as f:
        f.write(video_name + "\n")
    X_test_list.append(seq)
    y_test_list.append(label2idx[label])
    np.save(f"data/dataset_test_proc/{video_name}.npy", seq)

# Формирование датасетов
X_train = np.array(X_train_list)
y_train = np.array(y_train_list)
X_test = np.array(X_test_list)
y_test = np.array(y_test_list)




20%|█▉        | 4023/20400 [7:20:20<29:52:34,  6.57s/it]

Пример получившихся файлов находится в папке data

#### Загрузка ключевых точек из сохраненных данных

In [ ]:
train_proc_dir = 'data/dataset_train_proc'
test_proc_dir = 'data/dataset_test_proc'

X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

def find_npy_path(attachment_id, is_train=True):
    folder = train_proc_dir if is_train else test_proc_dir
    path = os.path.join(folder, f"{attachment_id}.npy")
    if os.path.exists(path):
      return path
    return None

for idx, row in tqdm(new_data.iterrows(), total=len(new_data)):
    attachment_id = row['attachment_id']

    video_path = find_npy_path(attachment_id, is_train=True)
    if video_path:
        seq = np.load(video_path)
        X_train_list.append(seq)
        continue

    video_path = find_npy_path(attachment_id, is_train=False)
    if video_path:
        seq = np.load(video_path)
        X_test_list.append(seq)
        continue

X_train = np.array(X_train_list)
X_test = np.array(X_test_list)

100%|██████████| 20/20 [00:00<00:00, 67.43it/s]


In [48]:
X_train[1]

array([-2.62276694e-01,  2.70918906e-01, -7.45401607e-08, ...,
       -1.00000000e+03, -1.00000000e+03, -1.00000000e+03], shape=(6552,))

#### Объединение всех файлов в 1

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class GestureDataset(Dataset):
    def __init__(self, data, transform=None):
        """
        data_dir: путь к папке, где лежат подпапки с классами
        class_names: список названий классов, например ['no-gesture', 'да', 'нет', ...]
        """
        self.data = []
        self.labels = []
        self.transform = transform

        for idx, row in tqdm(data.iterrows(), total=len(data)):
            attachment_id = row['attachment_id']
            
            video_path = find_npy_path(attachment_id, is_train=True)
            if video_path:
                seq = np.load(video_path)
                self.data.append(seq)
                self.labels.append(label2idx[row['text']])

        print(f"Загружено {len(self.data)} примеров")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]
        label = self.labels[idx]

        if self.transform:
            seq = self.transform(seq)

        return torch.from_numpy(seq).float(), torch.tensor(label, dtype=torch.long)


data_dir = train_dir 

full_dataset = GestureDataset(data=data)

train_idx, val_idx = train_test_split(
    range(len(full_dataset)),
    test_size=0.15,
    random_state=42
)

train_dataset = torch.utils.data.Subset(full_dataset, train_idx)
val_dataset   = torch.utils.data.Subset(full_dataset, val_idx)

# DataLoader'ы
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    drop_last=True         
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

In [ ]:
save_path = 'gesture_dataset_preprocessed.pth'

torch.save({
    'data': full_dataset.data,      # list of numpy arrays (уже нормализованные)
    'labels': full_dataset.labels,  # list of int
    'train_idx': train_idx,
    'val_idx': val_idx,
    'label2idx': label2idx,         # если нужно
}, save_path)

In [ ]:
def load_preprocessed_gesture_dataset(save_path='gesture_dataset_preprocessed.pth', 
                                      batch_size=64):
    
    saved = torch.load(save_path, weights_only=False)
    
    dataset = GestureDataset.__new__(GestureDataset)
    dataset.data = saved['data']
    dataset.labels = saved['labels']
    dataset.transform = None
    
    print(f"Загружено {len(dataset)} примеров из файла")
    
    # Subset'ы
    train_dataset = torch.utils.data.Subset(dataset, saved['train_idx'])
    val_dataset   = torch.utils.data.Subset(dataset, saved['val_idx'])
    
    # DataLoader'ы
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )
    
    return train_loader, val_loader, dataset